# 12 - Shared LSTM vs XGBoost Comparison - 23.03.2026

This notebook is the lean orchestration layer for the shared comparison harness in [`utils/model_family_comparison.py`](/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py).

It is designed for thesis-grade reruns:

- one shared data contract from the exported `setA` / `setB` feature tables,
- locked architectures only for the first fair comparison,
- resumable execution with pair-level checkpointing,
- incremental artifact saves during long runs,
- fast preflight checks plus an optional one-pair smoke test before overnight execution,
- and plot generation directly from saved CSV artifacts later if needed.

Main scope in this notebook:

- cumulative horizons only,
- locked LSTM `A6 = single 64 | L72`,
- locked XGBoost `P01_md3_lr003_mc5`,
- regimes: `per_building`, `pooled_same_buildings`,
- weather modes: `FW0`, `FW2`, `FW1`,
- metrics: `WAPE`, `RMSE`, `R2`, and `MAE`.

This is a regression workflow, so the default diagnostics are training curves, learning-rate curves, prediction traces, residual plots, and parity plots. No classification confusion matrix is used here.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()

# Colab-safe root detection. We only accept directories that contain this project's utils module.
candidates = [
    cwd,
    cwd / 'thesis-project',
    Path('/content/thesis-project'),
    Path('/content/Thesis/thesis-project'),
    Path('/content/drive/MyDrive/thesis-project'),
    Path('/content/drive/MyDrive/Thesis/thesis-project'),
]
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'utils' / 'model_family_comparison.py').exists():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate thesis-project root. In Colab, clone or copy the repository first, then re-run this cell."
    )

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('MPLCONFIGDIR', str((PROJECT_ROOT / '.mplconfig').resolve()))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Install runtime deps only when missing (mainly for fresh Colab runtimes).
import importlib.util
import subprocess

dep_to_package = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
    'xgboost': 'xgboost',
    'tensorflow': 'tensorflow',
}
missing_packages = [pkg for mod, pkg in dep_to_package.items() if importlib.util.find_spec(mod) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

from utils.model_family_comparison import (
    ExperimentConfig,
    build_artifact_status_table,
    build_base_frame_cache,
    build_comparison_manifest,
    ensure_results_dirs,
    load_saved_outputs,
    preview_feature_bundle,
    print_resume_diagnostics,
    run_full_comparison,
    run_sanity_check,
    validate_feature_contract,
    plot_horizon_curves,
    plot_learning_rate_curve_lstm,
    plot_metric_bars,
    plot_parity_scatter,
    plot_predictions_vs_actual,
    plot_residual_histogram,
    plot_residual_scatter,
    plot_training_curves_lstm,
    plot_training_curves_xgb,
)

config = ExperimentConfig(
    buildings=('U05', 'U06', 'LIB', 'U02B', 'SOC', 'U03'),
    horizons=(1, 2, 4, 6, 8, 12, 16, 20, 24, 36),
    regimes=('per_building', 'pooled_same_buildings'),
    weather_modes=('FW0', 'FW2', 'FW1'),
    modes=('M0', 'M2', 'M4'),
    lookback_hours=72,
    results_dir=PROJECT_ROOT / 'results' / 'model_family_comparison_23032026',
    lstm_architecture_id='A6',
    xgb_preset_id='P01_md3_lr003_mc5',
    random_seed=42,
)
paths = ensure_results_dirs(config)

config_df = pd.DataFrame([
    {'field': 'buildings', 'value': ', '.join(config.buildings)},
    {'field': 'horizons', 'value': ', '.join(str(h) for h in config.horizons)},
    {'field': 'regimes', 'value': ', '.join(config.regimes)},
    {'field': 'weather_modes', 'value': ', '.join(config.weather_modes)},
    {'field': 'modes', 'value': ', '.join(config.modes)},
    {'field': 'lookback_hours', 'value': str(config.lookback_hours)},
    {'field': 'train_end', 'value': str(config.train_end)},
    {'field': 'test_start', 'value': str(config.test_start)},
    {'field': 'results_dir', 'value': str(config.results_dir)},
    {'field': 'lstm_architecture_id', 'value': config.lstm_architecture_id},
    {'field': 'xgb_preset_id', 'value': config.xgb_preset_id},
])
display(Markdown('### Runtime configuration'))
display(config_df)


ModuleNotFoundError: No module named 'utils.model_family_comparison'

In [ ]:
# Resume: where CSVs are read from + how many pair slots are already complete (same rules as the harness).
print_resume_diagnostics(config)


--- resume diagnostics ---
cwd: /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project
results_dir (resolved): /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/model_family_comparison_23032026
  comparison_manifest.csv: exists=True data_rows=1260
  comparison_metrics.csv: exists=True data_rows=1308
  comparison_predictions.csv: exists=True data_rows=4371788
  comparison_coverage.csv: exists=True data_rows=654
  run_log.csv: exists=True data_rows=1118
pair slots complete: 278/630 (same rules as pair_is_complete)
first incomplete slot: [9/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=24
--- end resume diagnostics ---


In [ ]:
base_frames = build_base_frame_cache(config)
manifest_df = build_comparison_manifest(config)
contract_df = validate_feature_contract(config, base_frames=base_frames)
bundle, split_spec, preview_df = preview_feature_bundle(
    config,
    building='U05',
    mode='M4',
    weather_mode='FW2',
    horizon_h=24,
    base_frames=base_frames,
)

setA_cols = pd.DataFrame({'setA_first_40_cols': list(base_frames['U05']['setA'].columns[:40])})
setB_cols = pd.DataFrame({'setB_first_40_cols': list(base_frames['U05']['setB'].columns[:40])})
bundle_summary_df = pd.DataFrame([
    {'field': 'feature_set', 'value': bundle.model_ready['feature_set']},
    {'field': 'target_name', 'value': bundle.target_name},
    {'field': 'mode', 'value': bundle.model_ready['mode']},
    {'field': 'weather_mode', 'value': bundle.model_ready['weather_mode']},
    {'field': 'horizon_h', 'value': str(bundle.model_ready['horizon_h'])},
    {'field': 'n_dynamic_features', 'value': str(len(bundle.dynamic_feature_cols))},
    {'field': 'n_static_features', 'value': str(len(bundle.static_feature_cols))},
    {'field': 'train_issue_end', 'value': str(split_spec.train_issue_end)},
    {'field': 'train_issue_rows', 'value': str(int(split_spec.train_issue_mask.sum()))},
    {'field': 'test_issue_rows', 'value': str(int(split_spec.test_issue_mask.sum()))},
    {'field': 'valid_issue_rows', 'value': str(int(bundle.valid_issue_mask.sum()))},
    {'field': 'manifest_rows', 'value': str(int(len(manifest_df)))},
])

display(Markdown('### Feature-contract check'))
display(contract_df)
display(Markdown('### Shared feature-bundle preview (`U05`, `M4`, `FW2`, `h=24`)'))
display(bundle_summary_df)
display(preview_df)
display(Markdown('### Exported data-structure peek'))
display(setA_cols)
display(setB_cols)


### Feature-contract check

,building,setA_missing_cols,setB_missing_cols,missing_fw_horizons,static_complete_M4,rows_setA,rows_setB,status
0,LIB,,,,True,22954,22954,ok
1,SOC,,,,True,22954,22954,ok
2,U02B,,,,True,22925,22925,ok
3,U03,,,,True,22925,22925,ok
4,U05,,,,True,25206,25206,ok
5,U06,,,,True,25206,25206,ok


### Shared feature-bundle preview (`U05`, `M4`, `FW2`, `h=24`)

,field,value
0,feature_set,setB
1,target_name,target_cum_h24
2,mode,M4
3,weather_mode,FW2
4,horizon_h,24
5,n_dynamic_features,68
6,n_static_features,20
7,train_issue_end,2023-12-31 00:00:00
8,train_issue_rows,16428
9,test_issue_rows,8755


,datetime,building,heat_kwh,target_cum_h24,feat_is_heating_weather,feat_heat_obs,feat_outdoor_temp_c,feat_wind_ms,feat_solar_irradiance_wm2,feat_hour_sin,feat_hour_cos,valid_issue
0,2022-02-14 13:00:00,U05,2.000000,45.347619,1.0,2.000000,2.20,3.6,96.775830,-0.258819,-9.659258e-01,True
1,2022-02-14 14:00:00,U05,1.866667,45.347619,1.0,1.866667,2.05,4.0,53.199444,-0.500000,-8.660254e-01,True
2,2022-02-14 15:00:00,U05,2.000000,45.480952,1.0,2.000000,1.85,4.7,10.983611,-0.707107,-7.071068e-01,True
3,2022-02-14 16:00:00,U05,2.000000,44.880952,1.0,2.000000,1.85,5.8,0.063333,-0.866025,-5.000000e-01,True
4,2022-02-14 17:00:00,U05,2.000000,44.880952,1.0,2.000000,2.30,3.7,0.000000,-0.965926,-2.588190e-01,True
5,2022-02-14 18:00:00,U05,2.000000,44.880952,1.0,2.000000,2.50,4.1,0.000000,-1.000000,-1.836970e-16,True
6,2022-02-14 19:00:00,U05,1.866667,44.880952,1.0,1.866667,2.55,3.6,0.000000,-0.965926,2.588190e-01,True
7,2022-02-14 20:00:00,U05,2.000000,44.914286,1.0,2.000000,3.00,3.8,0.000000,-0.866025,5.000000e-01,True


### Exported data-structure peek

,setA_first_40_cols
0,datetime
1,feat_outdoor_temp_c
2,feat_windchill_c
3,feat_hdh_15c
4,feat_is_heating_weather
5,feat_temp_roll6h
6,feat_temp_roll12h
7,feat_temp_roll24h
8,feat_temp_diff1h
9,feat_temp_diff3h


,setB_first_40_cols
0,datetime
1,feat_outdoor_temp_c
2,feat_windchill_c
3,feat_hdh_15c
4,feat_is_heating_weather
5,feat_temp_roll6h
6,feat_temp_roll12h
7,feat_temp_roll24h
8,feat_temp_diff1h
9,feat_temp_diff3h


## Sanity Check And Run Controls

Before an overnight run, use the sanity-check cell below. It performs fast environment and feature-contract checks, and it can optionally run a one-pair smoke test into a separate `_sanity_smoke` results folder.

The full run cell is resumable and saves artifacts after each pair. That means you can stop and restart the notebook later, and you can regenerate plots directly from the saved CSV files without retraining.


In [ ]:
RUN_SANITY_CHECK = False
RUN_SANITY_SMOKE = True
REQUIRE_SANITY_PASS = True

sanity_df = pd.DataFrame()
if RUN_SANITY_CHECK:
    sanity_df = run_sanity_check(
        config,
        base_frames=base_frames,
        run_smoke=RUN_SANITY_SMOKE,
        smoke_building='U05',
        smoke_horizon_h=8,
        smoke_mode='M0',
        smoke_weather_mode='FW0',
    )
    display(Markdown('### Sanity-check results'))
    display(sanity_df)
else:
    display(Markdown('Sanity checks are disabled for this session.'))

blocked_sanity = False
if not sanity_df.empty and 'status' in sanity_df.columns:
    blocked_sanity = sanity_df['status'].astype(str).eq('block').any()

if REQUIRE_SANITY_PASS and blocked_sanity:
    blocked_checks = sanity_df.loc[sanity_df['status'].astype(str) == 'block', ['check', 'detail']]
    raise RuntimeError(
        'Sanity check has blocking issues. Fix them before launching the full matrix.\n'
        + blocked_checks.to_string(index=False)
    )


Sanity checks are disabled for this session.

In [ ]:
RUN_FULL_MATRIX = True
RESUME_EXISTING = True
SAVE_AFTER_EACH_PAIR = True
CONTINUE_ON_ERROR = True

if RUN_FULL_MATRIX:
    outputs = run_full_comparison(
        config,
        save_artifacts=True,
        verbose=True,
        resume_existing=RESUME_EXISTING,
        save_after_each_pair=SAVE_AFTER_EACH_PAIR,
        continue_on_error=CONTINUE_ON_ERROR,
    )
    display(Markdown('### Run completed'))
    display(outputs.run_log_df.tail(12))
else:
    outputs = None
    display(Markdown(
        'Set `RUN_FULL_MATRIX = True` to execute the shared comparison harness. '
        'Completed pairs will be reused when `RESUME_EXISTING = True`.'
    ))


[  1/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=1 | resume-skip
[  2/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=2 | resume-skip
[  3/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=4 | resume-skip
[  4/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=6 | resume-skip
[  5/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=8 | resume-skip
[  6/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=12 | resume-skip
[  7/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=16 | resume-skip
[  8/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=20 | resume-skip
[  9/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=24


I0000 00:00:1774386484.393900 2976536 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1774386484.394088 2976536 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


[ 10/630] regime=per_building scope=U05 mode=M0 weather=FW0 h=36
[ 11/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=1
[ 12/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=2
[ 13/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=4
[ 14/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=6
[ 15/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=8
[ 16/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=12
[ 17/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=16
[ 18/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=20
[ 19/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=24
[ 20/630] regime=per_building scope=U05 mode=M0 weather=FW2 h=36
[ 21/630] regime=per_building scope=U05 mode=M0 weather=FW1 h=1
[ 22/630] regime=per_building scope=U05 mode=M0 weather=FW1 h=2
[ 23/630] regime=per_building scope=U05 mode=M0 weather=FW1 h=4
[ 24/630] regime=per_building scope=U05 mode=M0 weather=FW1 h=6
[ 25/630] regime=per_building scop

/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:1294: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:1294: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)


[ 81/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=1
[ 82/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=2
[ 83/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=4
[ 84/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=6
[ 85/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=8
[ 86/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=12
[ 87/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=16
[ 88/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=20
[ 89/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=24
[ 90/630] regime=per_building scope=U05 mode=M4 weather=FW1 h=36


/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:1294: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)
/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/utils/model_family_comparison.py:1294: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  work_df["is_heating_eval"] = heating_mask(work_df).to_numpy(dtype=bool)


[ 91/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=1
[ 92/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=2
[ 93/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=4
[ 94/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=6
[ 95/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=8
[ 96/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=12
[ 97/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=16
[ 98/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=20
[ 99/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=24
[100/630] regime=per_building scope=U06 mode=M0 weather=FW0 h=36
[101/630] regime=per_building scope=U06 mode=M0 weather=FW2 h=1
[102/630] regime=per_building scope=U06 mode=M0 weather=FW2 h=2
[103/630] regime=per_building scope=U06 mode=M0 weather=FW2 h=4
[104/630] regime=per_building scope=U06 mode=M0 weather=FW2 h=6
[105/630] regime=per_building scope=U06 mode=M0 weather=FW2 h=8
[106/630] regime=per_building scope

In [ ]:
# outputs = None
saved_outputs = outputs if outputs is not None else load_saved_outputs(config, manifest_df=manifest_df)

manifest_df = saved_outputs.manifest_df.copy()
metrics_df = saved_outputs.comparison_metrics_df.copy()
summary_df = saved_outputs.comparison_summary_df.copy()
predictions_df = saved_outputs.comparison_predictions_df.copy()
coverage_df = saved_outputs.comparison_coverage_df.copy()
run_log_df = saved_outputs.run_log_df.copy()
lstm_history_df = saved_outputs.lstm_training_history_df.copy()
xgb_history_df = saved_outputs.xgb_eval_history_df.copy()
artifact_status_df = build_artifact_status_table(config)

display(Markdown('### Saved artifact overview'))
display(artifact_status_df)

if not run_log_df.empty:
    display(Markdown('### Latest run-log rows'))
    display(run_log_df.tail(12))


In [ ]:
FOCUS_REGIME = 'per_building'
FOCUS_MODE = 'M4'
FOCUS_WEATHER = 'FW2'
FOCUS_H = 24
FOCUS_BUILDING = 'U05'

if summary_df.empty or metrics_df.empty or predictions_df.empty:
    display(Markdown('Run the full matrix first, or load existing artifacts, before plotting.'))
else:
    display(Markdown('### Horizon- and building-level comparison plots'))
    plot_horizon_curves(
        summary_df,
        metric='wape_mean',
        regime=FOCUS_REGIME,
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        save_path=paths['plots'] / 'horizon_curve_wape.png',
    )
    plot_metric_bars(
        metrics_df,
        metric='wape_pct',
        regime=FOCUS_REGIME,
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        save_path=paths['plots'] / 'metric_bars_wape_h24.png',
    )

    display(Markdown('### Prediction and residual diagnostics'))
    plot_predictions_vs_actual(
        predictions_df,
        regime=FOCUS_REGIME,
        model_family='lstm',
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        building=FOCUS_BUILDING,
        save_path=paths['plots'] / 'lstm_predictions_vs_actual.png',
    )
    plot_predictions_vs_actual(
        predictions_df,
        regime=FOCUS_REGIME,
        model_family='xgboost',
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        building=FOCUS_BUILDING,
        save_path=paths['plots'] / 'xgb_predictions_vs_actual.png',
    )
    plot_residual_scatter(
        predictions_df,
        regime=FOCUS_REGIME,
        model_family='lstm',
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        building=FOCUS_BUILDING,
        save_path=paths['plots'] / 'lstm_residual_scatter.png',
    )
    plot_residual_histogram(
        predictions_df,
        regime=FOCUS_REGIME,
        model_family='xgboost',
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        building=FOCUS_BUILDING,
        save_path=paths['plots'] / 'xgb_residual_histogram.png',
    )
    plot_parity_scatter(
        predictions_df,
        regime=FOCUS_REGIME,
        model_family='lstm',
        mode=FOCUS_MODE,
        weather_mode=FOCUS_WEATHER,
        horizon_h=FOCUS_H,
        building=FOCUS_BUILDING,
        save_path=paths['plots'] / 'lstm_parity.png',
    )

    display(Markdown('### Training diagnostics'))
    lstm_run_id = f'{FOCUS_REGIME}__{FOCUS_BUILDING}__lstm__{FOCUS_MODE}__{FOCUS_WEATHER}__h{int(FOCUS_H):02d}'
    xgb_run_id = f'{FOCUS_REGIME}__{FOCUS_BUILDING}__xgboost__{FOCUS_MODE}__{FOCUS_WEATHER}__h{int(FOCUS_H):02d}'
    if not lstm_history_df.empty:
        plot_training_curves_lstm(
            lstm_history_df,
            run_id=lstm_run_id,
            save_path=paths['plots'] / 'lstm_training_curve.png',
        )
        plot_learning_rate_curve_lstm(
            lstm_history_df,
            run_id=lstm_run_id,
            save_path=paths['plots'] / 'lstm_learning_rate_curve.png',
        )
    if not xgb_history_df.empty:
        plot_training_curves_xgb(
            xgb_history_df,
            run_id=xgb_run_id,
            metric='rmse',
            save_path=paths['plots'] / 'xgb_eval_curve.png',
        )


In [ ]:
if summary_df.empty:
    display(Markdown('No saved summary is available yet. Run the matrix first to generate thesis-ready tables.'))
else:
    thesis_focus = summary_df[
        (summary_df['regime'] == 'per_building')
        & (summary_df['weather_mode'] == 'FW2')
    ].copy()
    thesis_pivot = thesis_focus.pivot_table(
        index=['mode', 'horizon_h'],
        columns='model_family',
        values=['wape_mean', 'rmse_mean', 'r2_mean', 'mae_mean'],
    )
    display(Markdown('### Thesis-facing comparison table (`per_building`, `FW2`)'))
    display(thesis_pivot)

    if not metrics_df.empty:
        win_table = metrics_df[
            (metrics_df['regime'] == 'per_building')
            & (metrics_df['weather_mode'] == 'FW2')
        ].copy()
        win_table['is_best_wape'] = win_table.groupby(['building', 'mode', 'horizon_h'])['wape_pct'].transform('min') == win_table['wape_pct']
        display(Markdown('### Building-level win table (`per_building`, `FW2`)'))
        display(win_table.head(24))

    if not run_log_df.empty:
        failed_rows = run_log_df.loc[run_log_df['status'].astype(str).ne('ok')].copy()
        display(Markdown('### Non-OK run-log rows'))
        display(failed_rows.tail(24) if not failed_rows.empty else pd.DataFrame({'status': ['all completed rows currently marked ok']}))
